In [1]:

from typing import Tuple, NamedTuple
import numpy as np
import copy
import pandas as pd
from scipy.special import logsumexp

"""Mixture model for collaborative filtering"""
class GaussianMixture(NamedTuple):
    """Tuple holding a gaussian mixture"""
    mu: np.ndarray  # (K, d) array - each row corresponds to a gaussian component mean
    var: np.ndarray  # (K, ) array - each row corresponds to the variance of a component
    p: np.ndarray  # (K, ) array = each row corresponds to the weight of a component

def run_matrix_completion_once(ratings_df: pd.DataFrame, K: int = 3):
    X, user_id_to_idx, hotel_id_to_idx, idx_to_user_id, idx_to_hotel_id = build_rating_matrix(ratings_df)
    mixture_init, post = initialise(X, K, seed=10)
    mixture_fit, post, ll = run(X, mixture_init, post)
    X_pred = fill_matrix(X, mixture_fit)
    return X_pred, mixture_fit, user_id_to_idx, hotel_id_to_idx, idx_to_user_id, idx_to_hotel_id

def build_rating_matrix(ratings_df: pd.DataFrame):
    """
    Build a user-by-hotel rating matrix X from a pandas DataFrame.

    Expected columns: 'user_id', 'hotel_id', 'rating'
    Returns:
        X: (n_users, n_hotels) rating matrix with 0 for missing ratings
        user_id_to_idx: dict mapping user_id -> row index
        hotel_id_to_idx: dict mapping hotel_id -> col index
        idx_to_user_id: np.array mapping row index -> user_id
        idx_to_hotel_id: np.array mapping col index -> hotel_id
    """
    # Get unique IDs (sorted for reproducibility)
    user_ids = np.sort(ratings_df["user_id"].unique())
    hotel_ids = np.sort(ratings_df["hotel_id"].unique())

    n_users = len(user_ids)
    n_hotels = len(hotel_ids)

    # Initialize matrix with zeros (missing = 0)
    X = np.zeros((n_users, n_hotels), dtype=np.float32)

    # Build index mappings
    user_id_to_idx = {uid: i for i, uid in enumerate(user_ids)}
    hotel_id_to_idx = {hid: j for j, hid in enumerate(hotel_ids)}

    # Fill in observed ratings
    for _, row in ratings_df.iterrows():
        u = user_id_to_idx[row["user_id"]]
        h = hotel_id_to_idx[row["hotel_id"]]
        X[u, h] = float(row["rating"])

    idx_to_user_id = user_ids
    idx_to_hotel_id = hotel_ids

    return X, user_id_to_idx, hotel_id_to_idx, idx_to_user_id, idx_to_hotel_id

def estep(X: np.ndarray, mixture: GaussianMixture) -> Tuple[np.ndarray, float]:
    """E-step: Softly assigns each datapoint to a gaussian component

        Args:
            X: (n, d) array holding the data, with incomplete entries (set to 0)
            mixture: the current gaussian mixture

        Returns:
            np.ndarray: (n, K) array holding the soft counts
                for all components for all examples
            float: log-likelihood of the assignment

        """
    n, d = X.shape
    K = mixture.mu.shape[0]

    # Storage for log p(z=j,x_i) and for the normalized posteriors
    log_pxz = np.zeros((n, K))
    post = np.zeros((n, K))
    ll = 0.0

    for i in range(n):
        observed = (X[i] != 0)
        num_obs = observed.sum()  # number of actually observed dims
        x_obs = X[i, observed]

        for j in range(K):
            # pull out the observed parts of mu and var
            mu_obs = mixture.mu[j, observed]
            var_jall = mixture.var[j]  # scalar (spherical) or 1‑d array
            var_obs = var_jall if np.isscalar(var_jall) else var_jall[observed]

            # stable logs
            log_pj = np.log(mixture.p[j] + 1e-16)
            log_var = np.log(var_obs + 1e-16)

            # Gaussian log‑density (diagonal/spherical)
            diff = x_obs - mu_obs
            term1 = np.sum(diff ** 2 / var_obs)
            term2 = num_obs * (log_var + np.log(2 * np.pi))
            log_gauss = -0.5 * (term1 + term2)

            log_pxz[i, j] = log_pj + log_gauss

        # log p(x_i) = log ∑_j p(x_i, z=j) in a stable way
        c = logsumexp(log_pxz[i])
        ll += c

        # posterior p(z=j | x_i) , exp( log p(x_i,z=j) − c )
        post[i] = np.exp(log_pxz[i] - c)

    return post, ll



def mstep(X: np.ndarray, post: np.ndarray, mixture: GaussianMixture,
          min_variance: float = .25) -> GaussianMixture:
    """M-step: Updates the gaussian mixture by maximizing the log-likelihood
    of the weighted dataset

    Args:
        X: (n, d) array holding the data, with incomplete entries (set to 0)
        post: (n, K) array holding the soft counts
            for all components for all examples
        mixture: the current gaussian mixture
        min_variance: the minimum variance for each gaussian

    Returns:
        GaussianMixture: the new gaussian mixture
    """
    n, d = X.shape
    n_hat = post.sum(axis=0)  # (K,)   effective cluster sizes
    K = n_hat.size

    # mixing proportions
    p_new = n_hat / n  # (K,)

    # means
    mu_new = mixture.mu.copy()  # start from old mu
    for k in range(K):
        for l in range(d):
            mask = X[:, l] != 0  # points that *observe* coordinate l
            denom = np.dot(post[mask, k], np.ones_like(post[mask, k]))
            if denom >= 1.0:  # at least one full point supports it
                mu_new[k, l] = np.dot(post[mask, k], X[mask, l]) / denom
            # else keep previous mu[k, l]

    # variances
    var_new = np.empty(K)
    for k in range(K):
        num, denom = 0.0, 0.0
        for u in range(n):
            obs = X[u] != 0  # boolean mask   C_u
            if not np.any(obs):
                continue  # user u has no ratings
            diff = X[u, obs] - mu_new[k, obs]
            num += post[u, k] * np.dot(diff, diff)
            denom += post[u, k] * obs.sum()  # |C_u|
        if denom == 0:
            var_new[k] = min_variance
        else:
            var_new[k] = max(num / denom, min_variance)

    return GaussianMixture(mu_new, var_new, p_new)


def run(X: np.ndarray, mixture: GaussianMixture,
        post: np.ndarray) -> Tuple[GaussianMixture, np.ndarray, float]:
    """Runs the mixture model

        Args:
            X: (n, d) array holding the data
            post: (n, K) array holding the soft counts
                for all components for all examples

        Returns:
            GaussianMixture: the new gaussian mixture
            np.ndarray: (n, K) array holding the soft counts
                for all components for all examples
            float: log-likelihood of the current assignment
        """
    epsilon: float = 1e-6
    # Deep copy the input data to avoid altering it in place
    X_copy = copy.deepcopy(X)
    # X_copy = X

    # Initialize the previous log-likelihood
    prev_ll = None
    ll = None

    while (prev_ll is None or ll - prev_ll > epsilon * abs(ll)):
        prev_ll = ll

        # E-step: Compute the soft assignments and log-likelihood
        post, ll = estep(X_copy, mixture)

        # M-step: Update the parameters of the Gaussian mixture model
        mixture = mstep(X_copy, post, mixture)

    return mixture, post, ll


def fill_matrix(X: np.ndarray, mixture: GaussianMixture) -> np.ndarray:
    """Fills an incomplete matrix according to a mixture model

    Args:
        X: (n, d) array of incomplete data (incomplete entries =0)
        mixture: a mixture of gaussians

    Returns
        np.ndarray: a (n, d) array with completed data
    """
    # Deep copy the input matrix to avoid modifying it in place
    X_pred = X.copy()

    # E-step: Compute the soft assignments (posterior probabilities)
    post, _ = estep(X, mixture)

    # Iterate over all users (rows) in the matrix
    n, d = X.shape
    for u in range(n):
        # Identify observed and missing values for user u
        observed = X[u] != 0
        missing = X[u] == 0

        # Predict the missing values by summing over the weighted means of each component
        for i in range(d):
            if missing[i]:  # Only fill missing values
                # Compute the weighted sum of the means for the missing value
                X_pred[u, i] = np.sum(post[u, :] * mixture.mu[:, i])

    return X_pred

def initialise(X: np.ndarray, K: int,
         seed: int = 0) -> Tuple[GaussianMixture, np.ndarray]:
    """Initializes the mixture model with random points as initial
    means and uniform assingments

    Args:
        X: (n, d) array holding the data
        K: number of components
        seed: random seed

    Returns:
        mixture: the initialized gaussian mixture
        post: (n, K) array holding the soft counts
            for all components for all examples

    """
    np.random.seed(seed)
    n, _ = X.shape
    p = np.ones(K) / K

    # select K random points as initial means
    mu = X[np.random.choice(n, K, replace=False)]
    var = np.zeros(K)
    # Compute variance
    for j in range(K):
        var[j] = ((X - mu[j])**2).mean()

    mixture = GaussianMixture(mu, var, p)
    post = np.ones((n, K)) / K

    return mixture, post

In [2]:
from sklearn.model_selection import train_test_split

data = pd.read_csv("/Users/angelothompson/Documents/University/Year_3/Period_2/RecSys/project/data/filtered/dense_user_item_ratings.csv")

data = data.drop_duplicates(subset=['user_id', 'item_id'])

data = data.rename(columns={"item_id": "hotel_id"})

# 70\% train, 10\% val, 20\% test
train_val_df, test_df = train_test_split(
    data,
    test_size=0.2,
    random_state=10,
    shuffle=True
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.125,   # 0.125 of 0.8 = 0.1
    random_state=10,
    shuffle=True
)

print(train_df.shape, val_df.shape, test_df.shape)

(980070, 3) (140011, 3) (280021, 3)


In [3]:
# Get predicted rating for a particular user and hotel:
def predict_single(user_id, hotel_id, X_pred, idx_to_user_id, idx_to_hotel_id):
    try:
        u_idx = np.where(idx_to_user_id == user_id)[0][0]
        h_idx = np.where(idx_to_hotel_id == hotel_id)[0][0]
    except IndexError:
        return None  # unseen user or hotel
    return X_pred[u_idx, h_idx]

In [4]:

def evaluate_test_accuracy(
    test_df: pd.DataFrame,
    X_pred: np.ndarray,
    user_id_to_idx: dict,
    hotel_id_to_idx: dict,
) -> dict:
    """
    Evaluate matrix-completion predictions on a held-out test set.

    test_df: DataFrame with columns ['user_id', 'hotel_id', 'rating']
    X_pred: completed rating matrix (n_users, n_hotels)
    user_id_to_idx: mapping from user_id -> row index in X_pred
    hotel_id_to_idx: mapping from hotel_id -> col index in X_pred

    Returns:
        dict with rmse, mae, accuracy (rounded), and number of evaluated points.
    """
    y_true = []
    y_pred = []

    for _, row in test_df.iterrows():
        uid = row["user_id"]
        hid = row["hotel_id"]
        rating = float(row["rating"])

        # skip users/hotels not seen in training
        if uid not in user_id_to_idx or hid not in hotel_id_to_idx:
            continue

        u_idx = user_id_to_idx[uid]
        h_idx = hotel_id_to_idx[hid]

        y_true.append(rating)
        y_pred.append(float(X_pred[u_idx, h_idx]))

    y_true = np.array(y_true, dtype=np.float32)
    y_pred = np.array(y_pred, dtype=np.float32)

    if len(y_true) == 0:
        return {"rmse": np.nan, "mae": np.nan, "accuracy": np.nan, "n": 0}

    # Regression-style metrics
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))

    # “Accuracy” by rounding predicted ratings to nearest integer
    y_pred_round = np.rint(y_pred)  # e.g. 3.7 -> 4
    accuracy = float(np.mean(y_pred_round == y_true))

    return {"rmse": rmse, "mae": mae, "accuracy": accuracy, "n": len(y_true)}

hyperparameter tuning on validation set

In [5]:
def evaluate_on_split(train_df, val_df, K):
    # train on train_df
    X_pred_train, mixture, user_id_to_idx, hotel_id_to_idx, idx_to_user_id, idx_to_hotel_id = run_matrix_completion_once(train_df, K=K)

    # build completed matrix in same index space
    X_pred_val = fill_matrix(
        build_rating_matrix(train_df)[0],  # X from train
        mixture
    )

    # evaluate on validation ratings, using the train mappings
    metrics = evaluate_test_accuracy(val_df, X_pred_val, user_id_to_idx, hotel_id_to_idx)
    return metrics, mixture, user_id_to_idx, hotel_id_to_idx, idx_to_user_id, idx_to_hotel_id

K_values = [2, 3, 4, 5, 6]
results = {}

best_K = None
best_rmse = np.inf
best_state = None

for K in K_values:
    metrics, mixture, user_id_to_idx, hotel_id_to_idx, idx_to_user_id, idx_to_hotel_id = \
        evaluate_on_split(train_df, val_df, K)
    results[K] = metrics
    print(f'K={K}: {metrics}')

    if metrics['rmse'] < best_rmse:
        best_rmse = metrics['rmse']
        best_K = K
        best_state = (mixture, user_id_to_idx, hotel_id_to_idx, idx_to_user_id, idx_to_hotel_id)

print(f'Best K: {best_K} with RMSE={best_rmse}')

K=2: {'rmse': np.float32(0.8448022), 'mae': np.float32(0.65639454), 'accuracy': 0.4606995164665633, 'n': 140011}
K=3: {'rmse': np.float32(0.8536609), 'mae': np.float32(0.65888226), 'accuracy': 0.46132089621529737, 'n': 140011}
K=4: {'rmse': np.float32(0.8537946), 'mae': np.float32(0.6589354), 'accuracy': 0.46130661162337244, 'n': 140011}
K=5: {'rmse': np.float32(0.8540705), 'mae': np.float32(0.6590253), 'accuracy': 0.4612923270314475, 'n': 140011}
K=6: {'rmse': np.float32(0.85493344), 'mae': np.float32(0.65929824), 'accuracy': 0.46134232310318474, 'n': 140011}
Best K: 2 with RMSE=0.84480220079422


test

In [6]:
# ratings_df with columns ['user_id', 'hotel_id', 'rating']
X_pred, mixture, user_id_to_idx, hotel_id_to_idx, idx_to_user_id, idx_to_hotel_id = run_matrix_completion_once(train_df, K=best_K)

metrics = evaluate_test_accuracy(test_df, X_pred, user_id_to_idx, hotel_id_to_idx)
print(metrics)

{'rmse': np.float32(0.8424643), 'mae': np.float32(0.6548526), 'accuracy': 0.4604226111613058, 'n': 280021}
